In [ ]:
import os
import sys
import numpy as np
import logging
import multiprocessing
from multiprocessing import Pool
import matplotlib.pyplot as plt
import pandas as pd
from astropy.coordinates import Angle
import astropy.units as u
import math
from userlist import *

################################################################################
#--- iSpec directory -------------------------------------------------------------
#ispec_dir = os.path.dirname(os.path.realpath(__file__)) + "/"
ispec_dir = "/Users/jiayue/iSpec/"
sys.path.insert(0, os.path.abspath(ispec_dir))
import ispec

#--- Change LOG level ----------------------------------------------------------
#LOG_LEVEL = "warning"
LOG_LEVEL = "info"
logger = logging.getLogger() # root logger, common for all
logger.setLevel(logging.getLevelName(LOG_LEVEL.upper()))
################################################################################


In [ ]:
from DataPreProcess import read_write_spectrum

# Read the spectrum

In [ ]:
spectra_target_path = input_spectra_path
spectra_target = read_write_spectrum(spectra_target_path, target, output_folder, unit_is_Angstrom)
#spectra_target["flux"] = spectra_target["flux"] * 1e11  # normalize flux
ispec.plot_spectra([spectra_target])

# Barycentric velocity and radial velocity correction

In [ ]:
from DataPreProcess import (
    parse_obs_time, parse_ra_to_hms, parse_dec_to_dms,
    determine_tellurics_shift_with_mask,
    calculate_barycentric_velocity,
    determine_radial_velocity_with_mask,
    apply_velocity_correction,   # optional
)

### 1. firstly run the radial velocity to check if it's the same as ref
### 2. (Optinal) If rv_test is NOT = ref rv, than do the barycentric velocity correction
### 3. Calculate radial velocity and do the rv correction

In [ ]:
# print the reference rv value
rv_ref = row_target["RV_km_s"]
print(f"Reference RV for {target}: ", rv_ref, "km/s")
rv_test, rv_err_test = determine_radial_velocity_with_mask(spectra_target)
print(f"Measured RV for {target}: ", rv_test, "km/s")

# infer if the rv is already corrected
if abs(rv_test - rv_ref) < 1.0:
    print("The spectrum is already corrected for radial velocity.")
    spectra_rv_corrected = apply_velocity_correction(spectra_target, rv_test)
    # plot spectra after correction
    ispec.plot_spectra([spectra_target, spectra_rv_corrected])
    # save the final corrected spectrum
    ispec.write_spectrum(spectra_rv_corrected, output_folder+"spectra_" + target + "_corrected.fits")
else:
    print("The spectrum is NOT corrected for radial velocity.")
    # ---- 1) parse observation metadata ----
    Obs_time = parse_obs_time(Obs_time_raw)            # -> [Y, M, D, h, m, s]
    target_ra = parse_ra_to_hms(target_ra_raw)         # -> [H, M, S]
    target_dec = parse_dec_to_dms(target_dec_raw)      # -> [deg, arcmin, arcsec]

    print("Obs_time:", Obs_time)
    print("RA (HMS):", target_ra)
    print("Dec (DMS):", target_dec)

    # ---- 2) telluric shift correction (optional but recommended) ----
    # first do telluric shift, then barycentric
    dv_tell, dv_tell_err, spectra_after_tell = determine_tellurics_shift_with_mask(spectra_target)

    # ---- 3) barycentric correction ----
    v_bary, spectra_after_bary = calculate_barycentric_velocity(
        spectra_after_tell, Obs_time, target_ra, target_dec)
    
    # ---- 4) RV measurement (after barycentric) ----
    # mask_file = ispec_dir + "input/linelists/CCF/HARPS_SOPHIE.G2.375_679nm/mask.lst"
    rv, rv_err = determine_radial_velocity_with_mask(spectra_after_bary)

    # ---- 5) apply RV correction (to rest frame) ----
    spectra_rv_corrected = apply_velocity_correction(spectra_after_bary, rv)

    # plot spectra after correction
    ispec.plot_spectra([spectra_target, spectra_after_bary, spectra_rv_corrected])
    # save the final corrected spectrum
    ispec.write_spectrum(spectra_rv_corrected, output_folder+"spectra_" + target + "_corrected.fits")

# Degrade the resolution

In [ ]:
# degrade the resolution
if from_resolution <= to_resolution:
    print("from_resolution is", from_resolution, ", which is lower than to_resolution", to_resolution)
else:
    degraded_spectrum = ispec.convolve_spectrum(spectra_rv_corrected, to_resolution, from_resolution=from_resolution)
    print("Degraded the spectrum from R=", from_resolution, " to R=", to_resolution)
    ispec.write_spectrum(degraded_spectrum, output_folder+"spectra_" + target + "_corrected_degraded_R" + str(to_resolution) + ".fits")

# Normalization

## 1. Normalize whole spectrum

In [ ]:
# read in the final spectrum for further analysis
if from_resolution <= to_resolution:
    # read the original and degraded spectra
    star_spectrum_target = ispec.read_spectrum(output_folder+"spectra_" + target + "_corrected.fits")
    print("No degradation applied, using the original corrected spectrum.")
else:
    star_spectrum_target = ispec.read_spectrum(output_folder+"spectra_" + target + "_corrected_degraded_R" + str(to_resolution) + ".fits")
    print("Using the degraded corrected spectrum.")

### 1.1 Normalize whole spectrum using template

In [ ]:
from DataPreProcess import normalize_whole_spectrum_with_template, plot_spectrum_with_normalization

In [ ]:
# Normalize the whole spectrum with a template
spectrum_norm, starcontnmodel = normalize_whole_spectrum_with_template(star_spectrum_target, from_resolution)

# Save normalized spectrum (keep your original naming style)
ispec.write_spectrum(spectrum_norm, output_folder + "spectra_" + target + "_normalized_template.fits")

# Plot diagnostics
plot_spectrum_with_normalization(
    raw_spectrum=star_spectrum_target,
    normalized_spectrum=spectrum_norm,
    continuum_model=starcontnmodel,
    starname=target,
    zoominrange1=[585, 595],
    zoominrange2=[588.7, 590]
)

In [ ]:
# --- Save normalized spectrum --------------------------------------------------
def save_segmented_normalized_spectra_figures(size, savefigpath, spectrum_norm, starname=target):
    """
    Save the normalized spectrum in segments as multiple image files.

    Parameters:
        size: Number of wavelength points to display per figure (e.g., 1000)
        savefigpath: Output directory to save figures (e.g., "HD39833spectraFigs/")
        spectrum_norm: iSpec-format normalized spectrum (must contain 'waveobs' and 'flux')
        starname: Star name used in figure titles and output filenames
    """
    wave = spectrum_norm['waveobs']
    flux = spectrum_norm['flux']
    num = int(wave.size / size)

    os.makedirs(savefigpath, exist_ok=True)
    print("Saving to:", savefigpath)
    print("Number of figures:", num)

    for i in range(num):
        fig = plt.figure(figsize=(15, 3), dpi=192)
        plt.plot(wave, flux, label='Spectrum', linewidth=1)
        plt.axhline(y=1, color='orange', linewidth=1, label='y=1')
        plt.xlim(wave[i * size], wave[(i + 1) * size - 1])
        plt.ylim(0.5,1.1)
        plt.title(f"{starname} after normalization, wave range: "
                  f"{round(wave[i * size], 3)} - {round(wave[(i + 1) * size - 1], 3)} nm")
        plt.savefig(os.path.join(savefigpath, f"{starname.replace(' ', '_')}_{i}.png"))
        plt.close(fig)

In [ ]:
#save_segmented_normalized_spectra_figures()

### 1.2 Normalize using continuum regions

In [ ]:
from DataPreProcess import normalize_spectrum_using_continuum_regions

In [ ]:
spectrum_norm_contrgs, starcontnmodel_contrgs = normalize_spectrum_using_continuum_regions(star_spectrum_target, to_resolution)
ispec.write_spectrum(spectrum_norm_contrgs, output_folder+"spectra_" + target + "_normalized_contrgs.fits")
plot_spectrum_with_normalization(star_spectrum_target, spectrum_norm_contrgs, starcontnmodel_contrgs, starname=target+"_contrgs", zoominrange1=[585, 595], zoominrange2=[588.7, 590])

## 3. Normalize in segments
- on Interactive interface